In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
import os
import zipfile
from google.colab import files
from collections import Counter
from sklearn.model_selection import train_test_split
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout, BatchNormalization
from tensorflow.keras.initializers import HeUniform, GlorotUniform
from tensorflow.keras.optimizers import Adam, SGD
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from keras.applications import MobileNet
from tensorflow.keras.layers import GlobalAveragePooling2D
from sklearn.metrics import accuracy_score
from sklearn.metrics import confusion_matrix, classification_report

Path

#Upload the Data

In [ ]:
uploaded = files.upload()
with zipfile.ZipFile('Braintumer Dataset.zip', 'r') as zip_ref:
    zip_ref.extractall()

In [ ]:
train_dir = '/content/Training'
test_dir = '/content/Testing'

In [ ]:
filepaths_train= []
label_train = []
folds = os.listdir(train_dir)
for fold in folds:
    foldpath = os.path.join(train_dir,fold)
    files = os.listdir(foldpath)
    for file in tqdm(files):
        filepath = os.path.join(foldpath,file)
        filepaths_train.append(filepath)
        label_train.append(fold)

In [ ]:
len(filepaths_train)

In [ ]:
dftrain = pd.DataFrame({
'filepaths_train' : filepaths_train,
'label_train' : label_train
})

In [ ]:
plt.pie(dftrain['label_train'].value_counts(),labels=dftrain['label_train'].value_counts().index,autopct='%.2f')
plt.title('Train Data Distribution')
plt.show()

In [ ]:
dftrain.head()

In [ ]:
dftrain['label_train'].value_counts()

#Read Test Data

In [ ]:
filepaths_valid = []
label_valid = []
folds = os.listdir(test_dir)
for fold in folds:
    foldpath = os.path.join(test_dir, fold)
    files = os.listdir(foldpath)
    for file in tqdm(files):
        filepath = os.path.join(foldpath, file)
        filepaths_valid.append(filepath)
        label_valid.append(fold)

In [ ]:
len(filepaths_valid)

In [ ]:
df_valid = pd.DataFrame({
    'filepaths_valid': filepaths_valid,
    'label_valid': label_valid
})

In [ ]:
df_valid.head()

In [ ]:
df_valid['label_valid'].value_counts()

#Train test split validation & test

In [ ]:
filepaths_valid, filepaths_test, label_valid, label_test = train_test_split(
    df_valid['filepaths_valid'], df_valid['label_valid'], test_size=0.2, random_state=42, stratify=df_valid['label_valid']
)

In [ ]:
df_test = pd.DataFrame({
    'filepaths_test': filepaths_test,
    'label_test': label_test
})

In [ ]:
print("Validation set size:", len(filepaths_valid))
print("Test set size:", len(filepaths_test))
print("Validation labels size:", len(label_valid))
print("Test labels size:", len(label_test))


In [ ]:
df_test.head()

#Modeling

#Rescaling & Normalizing Images

In [ ]:
train_datagen = ImageDataGenerator(rescale=1./255)
test_datagen = ImageDataGenerator(rescale=1./255)
valid_datagen = ImageDataGenerator(rescale=1./255)

Image Size & Batch Size

In [ ]:
IMAGE_SIZE = (224,224)
batch_size = 32

#Class Distribution

# Image Processing & Augmentation

In [ ]:
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=40,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    fill_mode='nearest'
)

valid_datagen = ImageDataGenerator(rescale=1./255)

# Loads & Feeds Data to Model

In [ ]:
train_gen = train_datagen.flow_from_dataframe(
    dftrain,
    x_col='filepaths_train',
    y_col='label_train',
    target_size=IMAGE_SIZE,
    shuffle=True,
    batch_size=batch_size
)

valid_gen = valid_datagen.flow_from_dataframe(
    df_valid,
    x_col='filepaths_valid',
    y_col='label_valid',
    target_size=IMAGE_SIZE,
    shuffle=False,
    batch_size=batch_size
)

test_gen = test_datagen.flow_from_dataframe(
    df_test,
    x_col='filepaths_test',
    y_col=None,
    target_size=IMAGE_SIZE,
    class_mode=None,
    shuffle=False,
    batch_size=batch_size
)

In [ ]:
print(f"Class Label :{train_gen.class_indices}")

In [ ]:
list(train_gen.class_indices.keys())

In [ ]:
class_labels = list(valid_gen.class_indices.keys())
class_counts_dict = Counter(valid_gen.classes)
class_counts = [class_counts_dict[i] for i in range(len(class_labels))]


plt.figure(figsize=(5, 4))
plt.bar(class_labels, class_counts, color='green')
plt.xlabel("Classes")
plt.ylabel("Number of Samples")
plt.title("Correct Class Distribution in Dataset")
plt.show()

In [ ]:
for batch_image, batch_label in train_gen:
    plt.figure(figsize=(10,10))
    for i in range (10):
        plt.subplot(5,5,i+1)
        plt.imshow(batch_image[i])
        plt.title(f"Class: {np.argmax(batch_label[i])}")
        plt.axis('off')
    break

#Using the MobileNet Model

In [ ]:
base = MobileNet(
    input_shape=(224,224,3),
    include_top=False,
    weights="imagenet"
)
base.trainable = False

model = Sequential([
    base,
    GlobalAveragePooling2D(),
    Dense(256, activation="relu"),
    Dense(128, activation="relu"),
    Dropout(0.5),
    Dense(4, activation="softmax")
])

model.summary()

In [ ]:
model.compile(optimizer=Adam(learning_rate=0.0001) , loss="categorical_crossentropy", metrics=["accuracy"])

In [ ]:
early_stopping = EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True
)

reduce_lr = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.2,
    patience=5,
    min_lr=1e-6
)

In [ ]:
history = model.fit(
    train_gen,
    validation_data=(valid_gen),
    epochs=40,
    callbacks=[early_stopping, reduce_lr]
)


train_accuracy = history.history['accuracy'][-1]
val_accuracy = history.history['val_accuracy'][-1]

In [ ]:
plt.figure(figsize=(12,4))
plt.subplot(1,2,1)
plt.plot(history.history['accuracy'])
plt.plot(history.history['val_accuracy'])
plt.title('Model Accuracy')
plt.ylabel('Accuracy')
plt.xlabel('Epoch')
plt.legend(['Train', 'Validation'], loc='upper left')

plt.subplot(1,2,2)
plt.plot(history.history['loss'])
plt.plot(history.history['val_loss'])
plt.title('Model Loss')
plt.ylabel('Loss')
plt.xlabel('Epoch')
plt.legend(['Train', 'Validation'], loc='upper left')
plt.tight_layout()
plt.show()

#Saving the model & making predictions

# Visualization the predictions by Confusion matrix

In [ ]:
class_indices = {'glioma': 0, 'meningioma': 1, 'notumor': 2, 'pituitary': 3}
index_to_class = {v: k for k, v in class_indices.items()}
class_names = list(class_indices.keys())


y_true = np.array(label_test)
preds = model.predict(test_gen)
y_pred = np.argmax(preds, axis=1)

if isinstance(y_true[0], str):
    y_true = np.array([class_indices[label] for label in y_true])

cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=class_names,
    yticklabels=class_names
)
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.title("Brain Tumor Classification - Confusion Matrix")
plt.tight_layout()
plt.show()


print("Classification Report:\n", classification_report(y_true, y_pred, target_names=class_names))

In [ ]:
test_accuracy = accuracy_score(y_true, y_pred)
print(f"Test Set Accuracy: {test_accuracy:.4f} ")

In [ ]:
model.save('Image_Detection_Model.h5')